# Facial Expressions → Event Features → Market Direction

This notebook demonstrates the end-to-end flow used in `test_face.py`, with added explanations and interactive controls.

Flow:
- Load per-frame facial expression estimates (arousal/valence/intensity).
- Aggregate frames to event-level features.
- Fetch daily prices with `yfinance`, compute next-day direction labels.
- Train a logistic model on face-derived features (optionally VLM features if available).
- Compare against a simple ARIMA baseline (price-only).

You can change the `ticker` and `event_dates` below and re-run. For illustration, we re-use the sample CSV shipped in the repo: `data/video_emotions/ywyBkewwcP0_sample.csv`.

In [1]:
# Setup: imports and dependency checks
import sys, subprocess, warnings
warnings.filterwarnings('ignore')

def ensure(pkg):
    try:
        __import__(pkg)
    except Exception:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for p in ['pandas','numpy','scikit-learn','statsmodels','yfinance','matplotlib','seaborn']:
    ensure(p)

import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

print('Environment ready.')

Installing scikit-learn ...
Installing yfinance ...


ImportError: cannot import name 'TypedDict'

## Configure Inputs

In [ ]:
# You can change these and re-run
ticker = 'DAL'
# Example event dates (use business days for cleaner labeling)
event_dates = pd.to_datetime(['2025-10-06','2025-10-07','2025-10-08','2025-10-09','2025-10-10'])

# Sample per-frame emotions CSV from the repo
video_csv_path = Path('data/video_emotions/ywyBkewwcP0_sample.csv')
assert video_csv_path.exists(), f'Missing sample CSV: {video_csv_path}'

# A notional call start time on each event date (doesn't affect daily labeling; used for timestamps only)
call_time_hhmm = (14, 30)  # 14:30 local time

print(f'Ticker: {ticker}')
print('Event dates:', ', '.join(d.strftime('%Y-%m-%d') for d in event_dates))
print('CSV:', video_csv_path)

## Load Per-frame Emotions and Build Faces Timeseries
We replicate the sample per-frame signals for each event, shifting timestamps to the configured call time on each event date.

In [ ]:
frames = pd.read_csv(video_csv_path)
frames = frames[frames.get('detected', True) == True].copy()
frames = frames[['timestamp_sec','arousal','valence','intensity']].astype(float)
frames.head()

In [ ]:
# Create faces_timeseries by repeating the per-frame curves for each event_id
event_ids = [f'evt{i+1}' for i in range(len(event_dates))]
faces_list = []
for eid, cdate in zip(event_ids, event_dates):
    ts0 = pd.Timestamp(cdate) + pd.Timedelta(hours=call_time_hhmm[0], minutes=call_time_hhmm[1])
    df = pd.DataFrame({
        'event_id': eid,
        'ts': ts0 + pd.to_timedelta(frames['timestamp_sec'].values, unit='s'),
        'valence': frames['valence'].values,
        'arousal': frames['arousal'].values,
        'intensity': frames['intensity'].values,
    })
    faces_list.append(df)
faces_ts = pd.concat(faces_list, ignore_index=True)
faces_ts.head()


## Aggregate to Event-level Features
We compute simple summary statistics per event: mean, std, slope (z-scored trend), mean absolute diff, and share above a small threshold.

In [ ]:
import numpy as np

def _slope(y):
    y = np.asarray(y)
    if y.size < 2: return 0.0
    x = np.arange(y.size)
    x = (x - x.mean()) / (x.std() + 1e-8)
    y = (y - y.mean()) / (y.std() + 1e-8)
    return float(np.dot(x, y) / (y.size - 1))

def _madiff(y):
    y = np.asarray(y)
    if y.size < 2: return 0.0
    return float(np.mean(np.abs(np.diff(y))))

def _share_above(y, thr=0.2):
    y = np.asarray(y)
    return float(np.mean(y > thr)) if y.size else 0.0

def build_face_features(faces_df):
    feats = []
    for eid, g in faces_df.groupby('event_id'):
        g = g.sort_values('ts')
        row = {'event_id': eid}
        for name in ['valence','arousal','intensity']:
            s = g[name].astype(float).values
            row[f'{name[:3]}_mean']   = float(np.mean(s)) if s.size else 0.0
            row[f'{name[:3]}_std']    = float(np.std(s))  if s.size else 0.0
            row[f'{name[:3]}_slope']  = _slope(s)
            row[f'{name[:3]}_madiff'] = _madiff(s)
            row[f'{name[:3]}_p20']    = _share_above(s, 0.2)
        feats.append(row)
    return pd.DataFrame(feats)

face_feats = build_face_features(faces_ts)
face_feats

## Create Events Table

In [ ]:
events = pd.DataFrame({
    'event_id': event_ids,
    'ticker': ticker,
    'call_start_utc': [pd.Timestamp(d) + pd.Timedelta(hours=call_time_hhmm[0], minutes=call_time_hhmm[1]) for d in event_dates],
    'call_date': event_dates.normalize(),
})
events


## Fetch Daily Prices with yfinance and Build Labels
We compute the next-day log return and a binary direction label per `call_date`.

In [ ]:
start = (events['call_date'].min() - pd.Timedelta(days=365)).strftime('%Y-%m-%d')
end   = (events['call_date'].max() + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
print('Downloading daily prices via yfinance ...')
px = yf.download(ticker, start=start, end=end, interval='1d', auto_adjust=False, progress=False, actions=False)
if px.empty:
    raise RuntimeError('No price data returned by yfinance. Try a different ticker or dates.')
px = px.reset_index().rename(columns={'Date':'date', 'Close':'close'})[['date','close']]
px['date'] = pd.to_datetime(px['date']).dt.tz_localize(None).dt.normalize()
px = px.sort_values('date')
# compute next-day log return
px['ret'] = np.log(px['close'].shift(-1) / px['close'])
labels = px.rename(columns={'date':'call_date'})[['call_date','ret']].copy()
labels['direction'] = (labels['ret'] > 0).astype(int)
labels.head()


## Merge Features, Split, and Train Logistic Model

In [ ]:
# Merge event-level features with events and price labels
df = events.merge(face_feats, on='event_id', how='inner')
df = df.merge(labels[['call_date','direction','ret']], on='call_date', how='left')
df = df.dropna(subset=['direction']).reset_index(drop=True)

# Column groups
face_cols = [c for c in df.columns if any(c.startswith(p) for p in ['val_','aro_','int_'])]
vlm_cols  = [c for c in df.columns if c.startswith('vlm_')]  # placeholder if you add VLM features

# Train/test split by time (last 20% as test)
df = df.sort_values('call_date')
split_idx = int(len(df)*0.8) if len(df) >= 10 else max(len(df)-2, 1)
train, test = df.iloc[:split_idx], df.iloc[split_idx:]
y_tr, y_te = train['direction'].values, test['direction'].values

transformers = [('face', StandardScaler(), face_cols)]
if len(vlm_cols) > 0:
    transformers.append(('vlm', Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=min(32, len(vlm_cols))))]), vlm_cols))
pre = ColumnTransformer(transformers=transformers, remainder='drop')
clf = LogisticRegression(max_iter=200, class_weight='balanced')
pipe = Pipeline([('pre', pre), ('clf', clf)])

pipe.fit(train, y_tr)
pred_proba = pipe.predict_proba(test)[:,1] if len(test) else np.array([])
pred_dir   = (pred_proba >= 0.5).astype(int) if pred_proba.size else np.array([])

def mda(y_true, y_pred_dir):
    return accuracy_score(y_true, y_pred_dir) if len(y_true) and len(y_pred_dir) else np.nan

face_mda   = mda(y_te, pred_dir)
face_auc   = roc_auc_score(y_te, pred_proba) if (len(np.unique(y_te))>1 and pred_proba.size) else np.nan
face_brier = brier_score_loss(y_te, pred_proba) if pred_proba.size else np.nan

print('=== Face(+VLM) model ===')
print(f'MDA:   {face_mda:.3f}' if not np.isnan(face_mda) else 'MDA:   n/a')
print(f'AUC:   {face_auc:.3f}' if not np.isnan(face_auc) else 'AUC:   n/a')
print(f'Brier: {face_brier:.3f}' if not np.isnan(face_brier) else 'Brier: n/a')

test_out = test.copy()
if pred_proba.size:
    test_out['pred_proba'] = pred_proba
    display(test_out[['event_id','call_date','direction','pred_proba']])
else:
    print('Not enough samples for a test split; adjust event_dates to include more events.')


## ARIMA Baseline (Price-only)
For each event, fit ARIMA(1,0,0) on prior trading days of returns and map the forecast to P(up). If not enough history, default to 0.5.

In [ ]:
from typing import Optional

def arima_direction_for_event(prices_df: pd.DataFrame, call_date: pd.Timestamp, window: int = 252) -> float:
    try:
        from statsmodels.tsa.arima.model import ARIMA
    except Exception:
        return 0.5
    px_t = prices_df.sort_values('date').copy()
    px_t['ret'] = np.log(px_t['close']).diff()
    hist = px_t[px_t['date'] < call_date].dropna(subset=['ret'])
    if len(hist) < 40:
        return 0.5
    hist = hist.tail(window)
    try:
        model = ARIMA(hist['ret'].values, order=(1,0,0))
        res = model.fit()
        fcast = float(res.forecast(1)[0])
        sigma = float(np.std(hist['ret'].values) + 1e-8)
        from scipy.stats import norm
        p_up = 1 - norm.cdf(0, loc=fcast, scale=sigma)
        return float(p_up)
    except Exception:
        return 0.5

# Prepare a daily prices frame matching our label build
px_daily = px[['date','close']].copy()

arima_p = [arima_direction_for_event(px_daily, d) for d in test['call_date']] if len(test) else []
arima_p = np.array(arima_p)
arima_dir = (arima_p >= 0.5).astype(int) if arima_p.size else np.array([])

arima_mda   = accuracy_score(y_te, arima_dir) if arima_dir.size else np.nan
arima_auc   = roc_auc_score(y_te, arima_p) if (arima_p.size and len(np.unique(y_te))>1) else np.nan
arima_brier = brier_score_loss(y_te, arima_p) if arima_p.size else np.nan

print('=== ARIMA baseline ===')
print(f'MDA:   {arima_mda:.3f}' if not np.isnan(arima_mda) else 'MDA:   n/a')
print(f'AUC:   {arima_auc:.3f}' if not np.isnan(arima_auc) else 'AUC:   n/a')
print(f'Brier: {arima_brier:.3f}' if not np.isnan(arima_brier) else 'Brier: n/a')

if arima_p.size and pred_proba.size:
    print('
=== Summary ===')
    print(f'Face(+VLM) MDA – ARIMA MDA = {float(face_mda - arima_mda):+.3f}')
else:
    print('
Summary: need a larger test set to compare.')


## Quick Visualization

In [ ]:
if 'pred_proba' in test_out.columns:
    fig, ax = plt.subplots(figsize=(6,3))
    sns.barplot(data=test_out, x='event_id', y='pred_proba', ax=ax, color='steelblue')
    ax.set_title('Predicted P(up) by event')
    ax.set_ylim(0,1)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    plt.show()
else:
    print('No predictions to plot (insufficient test samples).')


## Optional: Save the constructed CSVs
These match the filenames that `test_face.py` expects (faces_timeseries.csv, events.csv, prices.csv).

In [ ]:
save_csvs = False  # set True to write files next to this notebook
if save_csvs:
    faces_ts.to_csv('faces_timeseries.csv', index=False)
    events.to_csv('events.csv', index=False)
    px[['date','close']].to_csv('prices.csv', index=False)
    print('Wrote faces_timeseries.csv, events.csv, prices.csv')
else:
    print('Skipping CSV writes. Set save_csvs=True to export.')


---
### Notes
- You can add language-model (VLM) embeddings per event by merging columns prefixed with `vlm_` into `df` before training.
- For a minute-level ARIMAX demonstration aligned to the upload timestamp, see `source/run_event_arimax.py`.
- Real analyses should use many events and robust cross-validation; this notebook keeps a tiny set for clarity.